# read_data_v2 — Pipeline de filtrado del proyecto (sin leakage)

Reescritura del pipeline de `read_data.ipynb`. Cambios clave respecto a la versión original:

1. **Split temporal por día** (16 train / 4 test, últimos 4 días como test) en vez de `train_test_split` aleatorio sobre datos 5Hz.
2. **Outliers, OHE y poda de correlación** se ajustan **solo sobre train** y se aplican a test (sin information leakage).
3. **Columna `DATE`** preservada hasta el split; columnas auxiliares se descartan al final.
4. **Salida explícita en dos archivos** (`df_train.csv`, `df_test.csv`) más un `filter_metadata.json` reproducible.

> Limitación documentada: el filtro IQR sobre los targets `ACCEL_*` se mantiene (decisión de diseño) pero los umbrales se calculan sobre train. Esto sigue eliminando algunos picos reales tanto en train como en test, por lo que las métricas reportadas subestiman el error en escenarios extremos.


## Sección 0 — Imports y configuración

In [1]:
# Resolver raíz del repo de forma robusta independiente del CWD.
# Funciona tanto si el notebook se ejecuta desde la raíz como desde notebooks/.
import sys
from pathlib import Path

_here = Path.cwd()
if (_here / 'environment.yml').exists() and (_here / 'src').exists():
    REPO_ROOT = _here
elif (_here.parent / 'environment.yml').exists() and (_here.parent / 'src').exists():
    REPO_ROOT = _here.parent
else:
    raise RuntimeError(
        f'No se pudo resolver REPO_ROOT desde CWD={_here}. '
        'Esperaba encontrar environment.yml y src/ en el CWD o en su padre directo. '
        'Verifica que el notebook se ejecute desde la raíz del repo o desde notebooks/.'
    )

sys.path.insert(0, str(REPO_ROOT))

import json

import numpy as np
import pandas as pd

# Helpers extraídos a src/ en Sub-paso 3.2 (commit 977ce00).
# Las constantes metodológicas viven ahora en src/config.py.
from src.config import (
    TARGET_VARS,
    RANDOM_STATE,
    TOOL_VALIDATION,
    N_TEST_DAYS,
    HIGH_CARDINALITY_THRESHOLD,
    CORR_THRESHOLD,
    OUTLIER_STRATEGIES,
    KNOWN_CATEGORICALS,
)
from src.preprocessing import (
    load_h5_with_date,
    remove_consecutive_duplicates,
    split_by_date,
    fit_outlier_thresholds,
    apply_outlier_thresholds,
    decide_ohe_columns,
    fit_ohe,
    apply_ohe,
    fit_correlation_drop,
)

# --- Rutas (NO viven en src/config.py — dependen del lugar de ejecución) ---
H5_PATH = REPO_ROOT / 'Data' / 'upload_5hz_2020_6.h5'
OUT_DIR = REPO_ROOT

# Tracker para imprimir tabla resumen al final
stage_log = []

def log_stage(name, df):
    stage_log.append({'etapa': name, 'n_filas': len(df), 'n_cols': df.shape[1]})
    print(f'[{name}] shape = {df.shape}')


## Sección 1 — Carga H5 y filtros pre-split

Estos filtros no usan estadística agregada, por lo que pueden ejecutarse antes del split sin causar leakage.

In [2]:
# 1. Cargar H5 y construir DataFrame con columna DATE preservada
# (load_h5_with_date está en src.preprocessing — importado en cell 2)
df = load_h5_with_date(H5_PATH)
log_stage('1. carga H5 + DATE', df)
print('Días cargados:', sorted(df['DATE'].unique()))


Sensores: 48; subgrupos (días) detectados: 30


[1. carga H5 + DATE] shape = (12960000, 49)


Días cargados: ['2020-06-01', '2020-06-02', '2020-06-03', '2020-06-04', '2020-06-05', '2020-06-06', '2020-06-07', '2020-06-08', '2020-06-09', '2020-06-10', '2020-06-11', '2020-06-12', '2020-06-13', '2020-06-14', '2020-06-15', '2020-06-16', '2020-06-17', '2020-06-18', '2020-06-19', '2020-06-20', '2020-06-21', '2020-06-22', '2020-06-23', '2020-06-24', '2020-06-25', '2020-06-26', '2020-06-27', '2020-06-28', '2020-06-29', '2020-06-30']


In [3]:
# 2. Filtro 1 — herramienta (ACTIVE_TOOL_ACTUAL == 1018)
# Antes de filtrar, reportar discrepancia entre ACTIVE_TOOL y ACTIVE_TOOL_ACTUAL (el proyecto lo menciona)
if 'ACTIVE_TOOL' in df.columns and 'ACTIVE_TOOL_ACTUAL' in df.columns:
    cmp_mask = df[['ACTIVE_TOOL', 'ACTIVE_TOOL_ACTUAL']].dropna()
    if len(cmp_mask) > 0:
        disagree = (cmp_mask['ACTIVE_TOOL'] != cmp_mask['ACTIVE_TOOL_ACTUAL']).sum()
        pct = 100 * disagree / len(cmp_mask)
        print(f'Discrepancia ACTIVE_TOOL vs ACTIVE_TOOL_ACTUAL: {disagree}/{len(cmp_mask)} filas ({pct:.2f}%)')

assert 'ACTIVE_TOOL_ACTUAL' in df.columns, 'Falta columna ACTIVE_TOOL_ACTUAL'
df = df[df['ACTIVE_TOOL_ACTUAL'] == TOOL_VALIDATION].copy()
log_stage(f'2. filtro tool=={TOOL_VALIDATION}', df)

# Eliminar todas las variantes de ACTIVE_TOOL (ya cumplieron su rol como filtro)
tool_cols = [c for c in df.columns if c.startswith('ACTIVE_TOOL')]
df = df.drop(columns=tool_cols)
print(f'Eliminadas columnas tool: {tool_cols}')
log_stage('2b. drop ACTIVE_TOOL*', df)


Discrepancia ACTIVE_TOOL vs ACTIVE_TOOL_ACTUAL: 12535/8926723 filas (0.14%)
[2. filtro tool==1018] shape = (217939, 49)
Eliminadas columnas tool: ['ACTIVE_TOOL', 'ACTIVE_TOOL_ACTUAL', 'ACTIVE_TOOL_PLC', 'ACTIVE_TOOL_SPINDLE']
[2b. drop ACTIVE_TOOL*] shape = (217939, 45)


In [4]:
# 3. Filtro 2 — columnas constantes (un único valor o todo NaN)
const_cols = []
for col in df.columns:
    if col == 'DATE':
        continue
    nunique = df[col].dropna().nunique()
    if nunique <= 1:
        const_cols.append((col, nunique))

if const_cols:
    print(f'Eliminadas {len(const_cols)} columnas constantes:')
    for c, n in const_cols:
        print(f'  - {c}: nunique={n}')
    df = df.drop(columns=[c for c, _ in const_cols])
else:
    print('Sin columnas constantes.')
log_stage('3. drop constantes', df)


Eliminadas 17 columnas constantes:
  - AA5: nunique=1
  - AA6: nunique=1
  - AA7: nunique=1
  - ALARMS: nunique=1
  - CURR_AA5: nunique=1
  - CURR_AA6: nunique=1
  - CURR_AA7: nunique=1
  - EMERGENCY: nunique=1
  - FEEDRATE_OVERRIDE: nunique=1
  - NC_EMERGENCY: nunique=1
  - NC_OPERATING_MODE: nunique=1
  - NC_SINGLE_BLOCK: nunique=1
  - NC_START_OR_CYCLE_ON: nunique=1
  - PMC_OPERATING_MODE: nunique=1
  - PROG_END_M02: nunique=1
  - PROG_END_M30: nunique=1
  - SPINDLE_OVERRIDE: nunique=1
[3. drop constantes] shape = (217939, 28)


In [5]:
# 4. Filtro 3 — categóricas con cardinalidad > HIGH_CARDINALITY_THRESHOLD
# Heurística: una columna se considera categórica candidata si sus valores son todos
# enteros (object o int-like) y nunique está en (HIGH_CARDINALITY_THRESHOLD, 200].
# Si supera el umbral, se dropea (one-hot la haría explotar la dimensionalidad).
# CRÍTICO: las variables continuas conocidas (targets + las de OUTLIER_STRATEGIES como
# SPINDLE_LOAD) están protegidas del filtro aunque sus valores sean enteros.

CONTINUOUS_PROTECTED = set(OUTLIER_STRATEGIES.keys()) | set(TARGET_VARS)

drop_high_card = []
for col in df.columns:
    if col == 'DATE' or col in CONTINUOUS_PROTECTED:
        continue
    s = df[col].dropna()
    if len(s) == 0:
        continue
    nu = s.nunique()
    is_int_like = pd.api.types.is_integer_dtype(s) or (
        pd.api.types.is_float_dtype(s) and (s.dropna() % 1 == 0).all()
    )
    is_object = pd.api.types.is_object_dtype(s)
    # Detectar como categórica candidata
    if (is_int_like or is_object) and nu <= 200:
        if nu > HIGH_CARDINALITY_THRESHOLD:
            drop_high_card.append((col, nu))

if drop_high_card:
    print(f'Eliminadas {len(drop_high_card)} columnas categóricas de alta cardinalidad (> {HIGH_CARDINALITY_THRESHOLD} valores únicos):')
    for c, n in drop_high_card:
        print(f'  - {c}: nunique={n}')
    df = df.drop(columns=[c for c, _ in drop_high_card])
else:
    print(f'Sin columnas categóricas > {HIGH_CARDINALITY_THRESHOLD} valores únicos.')

log_stage('4. drop high-cardinality categ', df)


Eliminadas 1 columnas categóricas de alta cardinalidad (> 15 valores únicos):
  - PROG_NAME_SELECTED: nunique=158
[4. drop high-cardinality categ] shape = (217939, 27)


In [6]:
# 5. Filtro 4 — duplicados consecutivos (idle de la máquina = filas repetidas exactamente)
# (remove_consecutive_duplicates está en src.preprocessing — importado en cell 2)
before = len(df)
df = remove_consecutive_duplicates(df)
print(f'Eliminadas {before - len(df)} filas duplicadas consecutivas ({100*(before-len(df))/before:.2f}%)')
log_stage('5. drop consec duplicates', df)


Eliminadas 47 filas duplicadas consecutivas (0.02%)
[5. drop consec duplicates] shape = (217892, 27)


In [7]:
# 6. Filtro 5 — dropna sobre filas (cualquier NaN restante)
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Eliminadas {before - len(df)} filas con NaN ({100*(before-len(df))/before:.2f}%)')
log_stage('6. dropna', df)
print('Columnas finales pre-split:', list(df.columns))


Eliminadas 0 filas con NaN (0.00%)
[6. dropna] shape = (217892, 27)
Columnas finales pre-split: ['ACCEL_PEAK', 'ACCEL_RMS', 'ACCEL_RMS_FREQ', 'B', 'CURR_B', 'CURR_X', 'CURR_Y', 'CURR_Z', 'FEED_HOLD', 'JOG_OVERRIDE', 'LAST_M_CODE', 'PROG_BLOCK_NUM', 'PROG_NAME_ACTIVE', 'PROG_STATUS', 'RAPID_OVERRIDE', 'RAPID_TRAVERSING', 'SPEED_RMS', 'SPINDLE_ACTUAL_FEED', 'SPINDLE_ACTUAL_SPEED', 'SPINDLE_LOAD', 'SPINDLE_OVERRIDE_X105', 'TEMPERATURE', 'X', 'X105', 'Y', 'Z', 'DATE']


## Sección 2 — Split temporal por día

Las últimas `N_TEST_DAYS` fechas (cronológicamente) se reservan como test. Todas las estadísticas posteriores se calculan únicamente sobre train.

In [8]:
# 7. Listar fechas presentes y validar
dates_sorted = sorted(df['DATE'].unique())
n_dates = len(dates_sorted)
print(f'Fechas únicas tras filtros pre-split: {n_dates}')
for d in dates_sorted:
    n = (df['DATE'] == d).sum()
    print(f'  {d}: {n} filas')

assert n_dates >= N_TEST_DAYS + 1, f'Solo {n_dates} fechas, no se puede reservar {N_TEST_DAYS} para test'


Fechas únicas tras filtros pre-split: 20
  2020-06-01: 9585 filas
  2020-06-02: 7886 filas
  2020-06-03: 12385 filas
  2020-06-04: 10113 filas
  2020-06-08: 9264 filas
  2020-06-09: 10024 filas
  2020-06-10: 22866 filas
  2020-06-11: 10444 filas
  2020-06-12: 25764 filas
  2020-06-15: 3358 filas
  2020-06-16: 3633 filas
  2020-06-18: 14625 filas
  2020-06-19: 7471 filas
  2020-06-22: 8577 filas
  2020-06-23: 16542 filas
  2020-06-24: 7170 filas
  2020-06-25: 9345 filas
  2020-06-26: 6613 filas
  2020-06-29: 8672 filas
  2020-06-30: 13555 filas


In [9]:
# 8. Asignar últimas N_TEST_DAYS fechas a test
# (split_by_date está en src.preprocessing — importado en cell 2)
df_train, df_test, train_dates, test_dates = split_by_date(df, N_TEST_DAYS)

print(f'Train dates ({len(train_dates)}): {train_dates}')
print(f'Test  dates ({len(test_dates)}): {test_dates}')

print(f'\ndf_train: {df_train.shape}')
print(f'df_test:  {df_test.shape}')
print(f'Proporción test: {100*len(df_test)/(len(df_train)+len(df_test)):.1f}%')

log_stage('7. train tras split', df_train)
log_stage('7. test tras split', df_test)


Train dates (16): ['2020-06-01', '2020-06-02', '2020-06-03', '2020-06-04', '2020-06-08', '2020-06-09', '2020-06-10', '2020-06-11', '2020-06-12', '2020-06-15', '2020-06-16', '2020-06-18', '2020-06-19', '2020-06-22', '2020-06-23', '2020-06-24']
Test  dates (4): ['2020-06-25', '2020-06-26', '2020-06-29', '2020-06-30']

df_train: (179707, 27)
df_test:  (38185, 27)
Proporción test: 17.5%
[7. train tras split] shape = (179707, 27)
[7. test tras split] shape = (38185, 27)


## Sección 3 — Transformaciones fit-on-train

Outliers, one-hot encoding y poda por correlación: ajustar parámetros con `df_train` y aplicar a ambos conjuntos.

In [10]:
# 9. Filtro 6 — outliers (fit on train, apply to both)
# (fit_outlier_thresholds y apply_outlier_thresholds están en src.preprocessing — importados en cell 2)

outlier_thresholds = fit_outlier_thresholds(df_train, OUTLIER_STRATEGIES)
print('Thresholds calculados sobre train:')
for c, t in outlier_thresholds.items():
    print(f'  {c}: {t}')

# Snapshot de targets antes del filtro IQR para reportar pérdida
target_stats_before = {
    split: pd.DataFrame(d[TARGET_VARS].describe()).T[['min','max','mean','std']]
    for split, d in [('train', df_train), ('test', df_test)]
}

df_train, dropped_tr = apply_outlier_thresholds(df_train, outlier_thresholds)
df_test, dropped_te  = apply_outlier_thresholds(df_test, outlier_thresholds)
print(f'\nFilas eliminadas por outliers - train: {dropped_tr}, test: {dropped_te}')

target_stats_after = {
    split: pd.DataFrame(d[TARGET_VARS].describe()).T[['min','max','mean','std']]
    for split, d in [('train', df_train), ('test', df_test)]
}

print('\n== Comparación targets train ANTES vs DESPUÉS del filtro IQR ==')
print('ANTES:'); print(target_stats_before['train'])
print('DESPUÉS:'); print(target_stats_after['train'])
print('\n== Comparación targets test ANTES vs DESPUÉS ==')
print('ANTES:'); print(target_stats_before['test'])
print('DESPUÉS:'); print(target_stats_after['test'])

log_stage('8. outliers (train)', df_train)
log_stage('8. outliers (test)', df_test)


Thresholds calculados sobre train:
  ACCEL_PEAK: {'method': 'iqr', 'low': np.float64(-1161.0158031084097), 'high': np.float64(1618.4778359471534)}
  ACCEL_RMS: {'method': 'iqr', 'low': np.float64(-495.2460147288381), 'high': np.float64(686.7027100251645)}
  ACCEL_RMS_FREQ: {'method': 'iqr', 'low': np.float64(-562.7196662280024), 'high': np.float64(839.8230951659533)}
  B: {'method': 'domain', 'low': 0, 'high': 360}
  CURR_B: {'method': 'winsorize', 'low': np.float64(-7.99), 'high': np.float64(9.96)}
  CURR_X: {'method': 'winsorize', 'low': np.float64(-56.52), 'high': np.float64(55.86880000000005)}
  CURR_Y: {'method': 'winsorize', 'low': np.float64(0.79), 'high': np.float64(46.814700000000016)}
  CURR_Z: {'method': 'winsorize', 'low': np.float64(-137.39), 'high': np.float64(138.61)}
  SPEED_RMS: {'method': 'winsorize', 'low': np.float64(0.0873971899272874), 'high': np.float64(11.134784407913694)}
  SPINDLE_ACTUAL_FEED: {'method': 'percentile', 'low': np.float64(0.0), 'high': np.float64


Filas eliminadas por outliers - train: 7575, test: 2353

== Comparación targets train ANTES vs DESPUÉS del filtro IQR ==
ANTES:
                      min          max        mean         std
ACCEL_PEAK      14.976380  6178.227249  282.175485  343.555742
ACCEL_RMS        6.514279  3961.910326  121.515177  164.476495
ACCEL_RMS_FREQ  11.508632  3920.488163  163.021241  200.172247
DESPUÉS:
                      min          max        mean         std
ACCEL_PEAK      14.976380  1618.332182  259.514784  276.383445
ACCEL_RMS        6.514279   686.642005  110.172907  125.605406
ACCEL_RMS_FREQ  11.508632   839.433378  149.995988  160.112316

== Comparación targets test ANTES vs DESPUÉS ==
ANTES:
                      min          max        mean         std
ACCEL_PEAK      14.846482  2683.079583  298.070275  432.091198
ACCEL_RMS        6.584354  1338.418649  120.972395  172.205893
ACCEL_RMS_FREQ  11.733797  1081.169576  157.169789  213.454667
DESPUÉS:
                      min          max   

In [11]:
# 10. Filtro 7 — One-hot encoding (fit on train, apply to both)
# (decide_ohe_columns, fit_ohe, apply_ohe están en src.preprocessing — importados en cell 2)

# --- Auditoría preliminar de RAPID_TRAVERSING (preservada para trazabilidad ---
#     del proyecto; la decisión final la toma decide_ohe_columns) ---
preliminar = [c for c in KNOWN_CATEGORICALS if c in df_train.columns]
print(f'Columnas a OHE-ar (preliminar, sin auditar RAPID_TRAVERSING): {preliminar}')

if 'RAPID_TRAVERSING' in df_train.columns:
    rt_unique = sorted(df_train['RAPID_TRAVERSING'].unique())
    print(f'RAPID_TRAVERSING valores únicos en train: {rt_unique}')
    if len(rt_unique) > 2:
        print('  -> NO es binaria, se añade al OHE')
    else:
        print('  -> binaria, se mantiene como numérica')

# --- Decisión final delegada a src.preprocessing.decide_ohe_columns ---
ohe_cols = decide_ohe_columns(df_train, KNOWN_CATEGORICALS)
print(f'Columnas a OHE-ar (final): {ohe_cols}')

# --- Ajuste y aplicación ---
ohe = fit_ohe(df_train, ohe_cols)
ohe_categories = {col: list(cats) for col, cats in zip(ohe_cols, ohe.categories_)}
print(f'\nCategorías ajustadas en train: {ohe_categories}')

df_train = apply_ohe(df_train, ohe, ohe_cols)
df_test  = apply_ohe(df_test, ohe, ohe_cols)

print(f'\nTrain post-OHE: {df_train.shape}')
print(f'Test  post-OHE: {df_test.shape}')
log_stage('9. OHE (train)', df_train)
log_stage('9. OHE (test)', df_test)


Columnas a OHE-ar (preliminar, sin auditar RAPID_TRAVERSING): ['JOG_OVERRIDE', 'SPINDLE_OVERRIDE_X105', 'X105']
RAPID_TRAVERSING valores únicos en train: [np.float64(0.0), np.float64(1.0)]
  -> binaria, se mantiene como numérica
Columnas a OHE-ar (final): ['JOG_OVERRIDE', 'SPINDLE_OVERRIDE_X105', 'X105']

Categorías ajustadas en train: {'JOG_OVERRIDE': [np.float64(29.0), np.float64(47.0)], 'SPINDLE_OVERRIDE_X105': [np.float64(50.0), np.float64(60.0), np.float64(70.0), np.float64(80.0), np.float64(90.0), np.float64(100.0), np.float64(120.0), np.float64(420.0)], 'X105': [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(7.0), np.float64(37.0)]}

Train post-OHE: (172132, 39)
Test  post-OHE: (35832, 39)
[9. OHE (train)] shape = (172132, 39)
[9. OHE (test)] shape = (35832, 39)


In [12]:
# 11. Filtro 8 — poda de correlación > CORR_THRESHOLD (fit on train, conserva la de mayor varianza)
# (fit_correlation_drop está en src.preprocessing — importado en cell 2)

# Excluir DATE (no numérica) y targets para no desordenarlos
corr_dropped, corr_pairs = fit_correlation_drop(
    df_train, CORR_THRESHOLD, exclude=set(['DATE'] + TARGET_VARS)
)
print(f'Pares con |corr| > {CORR_THRESHOLD}:')
for i, j, v in corr_pairs:
    print(f'  {i}  vs  {j}  -> {v:.4f}')
print(f'\nColumnas eliminadas (la de menor varianza en cada par): {corr_dropped}')

df_train = df_train.drop(columns=corr_dropped)
df_test  = df_test.drop(columns=corr_dropped)
log_stage('10. corr-prune (train)', df_train)
log_stage('10. corr-prune (test)', df_test)


Pares con |corr| > 0.95:
  SPINDLE_OVERRIDE_X105_60.0  vs  X105_1.0  -> 1.0000
  SPINDLE_OVERRIDE_X105_70.0  vs  X105_2.0  -> 1.0000
  SPINDLE_OVERRIDE_X105_80.0  vs  X105_3.0  -> 1.0000
  SPINDLE_OVERRIDE_X105_90.0  vs  X105_4.0  -> 1.0000
  SPINDLE_OVERRIDE_X105_100.0  vs  X105_5.0  -> 1.0000
  SPINDLE_OVERRIDE_X105_120.0  vs  X105_7.0  -> 1.0000
  SPINDLE_OVERRIDE_X105_420.0  vs  X105_37.0  -> 1.0000

Columnas eliminadas (la de menor varianza en cada par): ['X105_1.0', 'X105_2.0', 'X105_4.0', 'X105_3.0', 'X105_37.0', 'X105_5.0', 'X105_7.0']
[10. corr-prune (train)] shape = (172132, 32)
[10. corr-prune (test)] shape = (35832, 32)


## Sección 4 — Salida

Guardar `df_train.csv`, `df_test.csv` y `filter_metadata.json`. El metadata permite a los notebooks de modelos auditar el pipeline.

In [13]:
# 12. Salida final
# Confirmar columnas idénticas y mismas variables target presentes
assert list(df_train.columns) == list(df_test.columns), 'Columnas train/test no coinciden'
for t in TARGET_VARS:
    assert t in df_train.columns and t in df_test.columns, f'Falta target {t}'

# Guardar CSVs
train_path = OUT_DIR / 'df_train.csv'
test_path  = OUT_DIR / 'df_test.csv'
df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)
print(f'Guardado {train_path}: {df_train.shape}')
print(f'Guardado {test_path}:  {df_test.shape}')

# Metadata para reproducibilidad
metadata = {
    'tool_validation': TOOL_VALIDATION,
    'n_test_days': N_TEST_DAYS,
    'high_cardinality_threshold': HIGH_CARDINALITY_THRESHOLD,
    'corr_threshold': CORR_THRESHOLD,
    'random_state': RANDOM_STATE,
    'train_dates': train_dates,
    'test_dates': test_dates,
    'target_vars': TARGET_VARS,
    'final_columns': list(df_train.columns),
    'outlier_thresholds': {
        c: {k: (float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v)
            for k, v in t.items()}
        for c, t in outlier_thresholds.items()
    },
    'ohe_columns': ohe_cols,
    'ohe_categories': {k: [str(x) for x in v] for k, v in ohe_categories.items()},
    'correlation_dropped': corr_dropped,
    'correlation_pairs_above_threshold': [
        {'col_a': i, 'col_b': j, 'abs_corr': float(v)} for i, j, v in corr_pairs
    ],
}
meta_path = OUT_DIR / 'filter_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f'Guardado {meta_path}')


Guardado <REPO_ROOT>/df_train.csv: (172132, 32)
Guardado <REPO_ROOT>/df_test.csv:  (35832, 32)
Guardado <REPO_ROOT>/filter_metadata.json


In [14]:
# 13. Tabla resumen del pipeline
summary = pd.DataFrame(stage_log)
print('\n===== RESUMEN DEL PIPELINE =====')
print(summary.to_string(index=False))



===== RESUMEN DEL PIPELINE =====
                         etapa  n_filas  n_cols
            1. carga H5 + DATE 12960000      49
          2. filtro tool==1018   217939      49
         2b. drop ACTIVE_TOOL*   217939      45
            3. drop constantes   217939      28
4. drop high-cardinality categ   217939      27
     5. drop consec duplicates   217892      27
                     6. dropna   217892      27
           7. train tras split   179707      27
            7. test tras split    38185      27
           8. outliers (train)   172132      27
            8. outliers (test)    35832      27
                9. OHE (train)   172132      39
                 9. OHE (test)    35832      39
        10. corr-prune (train)   172132      32
         10. corr-prune (test)    35832      32


## Sección 5 — Verificación

In [15]:
# Asserts del plan
assert df_train['DATE'].max() < df_test['DATE'].min(), 'Solapamiento temporal'
assert list(df_train.columns) == list(df_test.columns)
assert all(t in df_train.columns for t in TARGET_VARS)
print('✅ Assertions OK')
print(f'  - Última fecha train: {df_train["DATE"].max()}')
print(f'  - Primera fecha test: {df_test["DATE"].min()}')
print(f'  - {len(df_train.columns)} columnas en ambos splits')


✅ Assertions OK
  - Última fecha train: 2020-06-24
  - Primera fecha test: 2020-06-25
  - 32 columnas en ambos splits


In [16]:
# Smoke test: RandomForest sobre ACCEL_PEAK
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Quitar DATE y targets para construir X
feature_cols = [c for c in df_train.columns if c != 'DATE' and c not in TARGET_VARS]
target = 'ACCEL_PEAK'

X_tr = df_train[feature_cols].values
y_tr = df_train[target].values
X_te = df_test[feature_cols].values
y_te = df_test[target].values

print(f'Features: {len(feature_cols)} cols')
print(f'X_tr: {X_tr.shape}, X_te: {X_te.shape}')

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)

print(f'\n== Smoke test RF | target={target} | split temporal ==')
print(f'MAE  : {mean_absolute_error(y_te, y_pred):.4f}')
print(f'MSE  : {mean_squared_error(y_te, y_pred):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_te, y_pred)):.4f}')
print(f'R²   : {r2_score(y_te, y_pred):.4f}')

print('\nReferencia proyecto (split aleatorio + leakage):')
print('  ACCEL_PEAK sin carga: R² = 0.4638 / con carga: R² = 0.9153')
print('Es esperable que el R² actual sea menor — eso es honestidad estadística.')


Features: 28 cols
X_tr: (172132, 28), X_te: (35832, 28)



== Smoke test RF | target=ACCEL_PEAK | split temporal ==
MAE  : 70.4423
MSE  : 18996.2066
RMSE : 137.8267
R²   : 0.8420

Referencia anterior (split aleatorio + leakage):
  ACCEL_PEAK sin carga: R² = 0.4638 / con carga: R² = 0.9153
Es esperable que el R² actual sea menor — eso es honestidad estadística.
